# Defender xPoints breakdown

Standalone notebook: top 10 defenders by expected points, split by the point-type each xPoints comes from (goals, assists, clean sheets, defensive contribution, appearance points, bonus points, goals-conceded penalty).

Run jupyter from the repo root so `import fpl_v2` resolves (`uv run jupyter lab`).

In [1]:
# Ensure the repo root (the dir containing fpl_v2/) is importable, whatever the launch dir.
import sys, pathlib
root = pathlib.Path.cwd()
while not (root / 'fpl_v2').is_dir() and root != root.parent:
    root = root.parent
if str(root) not in sys.path:
    sys.path.insert(0, str(root))

In [2]:
import plotly.graph_objects as go

from fpl_v2 import pipeline

[07/26/26 21:30:57] INFO     No custom team name replacements found. You can configure these in       ]8;id=12489830;file:///home/peter/coding/football_stuff/.venv/lib/python3.12/site-packages/soccerdata/_config.py\_config.py]8;;\:]8;id=12489831;file:///home/peter/coding/football_stuff/.venv/lib/python3.12/site-packages/soccerdata/_config.py#91\91]8;;\
                             /home/peter/soccerdata/config/teamname_replacements.json.                             

                    INFO     No custom league dict found. You can configure additional leagues in    ]8;id=12489837;file:///home/peter/coding/football_stuff/.venv/lib/python3.12/site-packages/soccerdata/_config.py\_config.py]8;;\:]8;id=12489838;file:///home/peter/coding/football_stuff/.venv/lib/python3.12/site-packages/soccerdata/_config.py#189\189]8;;\
                             /home/peter/soccerdata/config/league_dict.json.                                       

In [3]:
forecast = pipeline.build_forecast()  # cached data; pipeline.build_forecast(refresh=True) to re-pull live

defenders = forecast[forecast['position'] == 'DEF'].sort_values('xPoints', ascending=False).head(10)
defenders[['web_name', 'team_name', 'xG', 'xAG', 'xClean', 'xBadGames',
           'expected_defcon_points', 'expected_appearance_points', 'expected_bonus_points', 'xPoints']]

                    INFO     Saving cached data to                                                   ]8;id=12489845;file:///home/peter/coding/football_stuff/.venv/lib/python3.12/site-packages/soccerdata/_common.py\_common.py]8;;\:]8;id=12489846;file:///home/peter/coding/football_stuff/.venv/lib/python3.12/site-packages/soccerdata/_common.py#250\250]8;;\
                             /home/peter/coding/football_stuff/fpl_v2/data/raw/understat_match                     

[2026-07-26 21:30:57] INFO     TLSLibrary:_load_library:397 - Successfully loaded TLS library: /home/peter/coding/football_stuff/.venv/lib/python3.12/site-packages/tls_requests/bin/tls-client-xgo-1.13.1-linux-amd64.so


                    INFO     Successfully loaded TLS library:                                      ]8;id=12489853;file:///home/peter/coding/football_stuff/.venv/lib/python3.12/site-packages/tls_requests/models/libraries.py\libraries.py]8;;\:]8;id=12489854;file:///home/peter/coding/football_stuff/.venv/lib/python3.12/site-packages/tls_requests/models/libraries.py#397\397]8;;\
                             /home/peter/coding/football_stuff/.venv/lib/python3.12/site-packages/                 
                             tls_requests/bin/tls-client-xgo-1.13.1-linux-amd64.so                                 

,web_name,team_name,xG,xAG,xClean,xBadGames,expected_defcon_points,expected_appearance_points,expected_bonus_points,xPoints
0,Gabriel,Arsenal,2.94,1.75,22.0,3.0,25.740922,62.0,30.0,208.978875
241,Virgil,Liverpool,3.77,1.44,15.0,8.0,33.988759,76.0,10.0,198.928759
373,Senesi,Spurs,1.53,4.72,10.0,7.0,45.847120,74.0,14.0,188.913435
271,Guéhi,Man City,4.05,2.37,12.0,5.0,19.341800,70.0,14.0,174.357064
371,Van Hecke,Spurs,3.31,1.57,10.0,7.0,34.828128,72.0,8.0,170.371812
52,Truffert,Bournemouth,1.35,3.07,11.0,9.0,21.454273,76.0,19.0,168.334448
180,Tarkowski,Everton,2.53,2.10,7.0,9.0,41.601834,74.0,12.0,167.581834
1,J.Timber,Arsenal,4.71,1.53,22.0,3.0,2.743052,56.0,9.0,161.534573
325,Thiaw,Newcastle,4.80,0.96,9.0,9.0,26.034137,67.0,12.0,160.106242
154,Lacroix,Crystal Palace,2.45,0.83,7.0,7.0,43.652061,69.0,11.0,159.785044


## Chart

`xpoints.breakdown` (folded into `pipeline.build_forecast`) splits `xPoints` into `goal_points`, `assist_points`, `clean_points`, `defcon_points`, `appearance_points`, `bonus_points` and `conceded_points`, which sum back to `xPoints`. `conceded_points` is the only negative term (the goals-conceded penalty), so it draws to the left of zero — everything else stacks to the right.

In [4]:
# Component -> (column, legend label, categorical color).
# Colors are the first seven slots of the validated categorical palette (fixed order,
# never cycled), assigned in the same order the traces stack. conceded_points is the
# only negative value, so it draws left of zero — kept last so its notch sits right
# at the bar's tail, just before the net-total label.
COMPONENTS = [
    ('goal_points', 'Goals', '#2a78d6'),
    ('assist_points', 'Assists', '#eb6834'),
    ('clean_points', 'Clean sheets', '#1baf7a'),
    ('defcon_points', 'Defensive contribution', '#eda100'),
    ('appearance_points', 'Appearance points', '#e87ba4'),
    ('bonus_points', 'Bonus points', '#4a3aa7'),
    ('conceded_points', 'Goals-conceded penalty', '#008300'),
]

SURFACE = '#fcfcfb'
GRIDLINE = '#e1e0d9'
AXIS = '#c3c2b7'
INK_PRIMARY = '#0b0b0b'
INK_SECONDARY = '#52514e'
INK_MUTED = '#898781'

# Horizontal stacked bars plot bottom-to-top in row order, so sort ascending
# to put the highest total at the top.
plot_df = defenders.sort_values('xPoints', ascending=True).reset_index(drop=True)

In [5]:
fig = go.Figure()
for col, label, color in COMPONENTS:
    fig.add_trace(go.Bar(
        y=plot_df['web_name'],
        x=plot_df[col],
        name=label,
        orientation='h',
        marker=dict(color=color, line=dict(color=SURFACE, width=1.5)),
        hovertemplate=f'%{{y}}<br>{label}: %{{x:.1f}} pts<extra></extra>',
    ))

# Direct label: net total, via a zero-width trailing bar + textposition='outside' so
# Plotly computes the padding past the bar tip itself, rather than a manual x-offset.
fig.add_trace(go.Bar(
    y=plot_df['web_name'], x=[0] * len(plot_df), orientation='h',
    marker=dict(color='rgba(0,0,0,0)'),
    text=[f'{t:.0f}' for t in plot_df['xPoints']],
    textposition='outside', textfont=dict(color=INK_SECONDARY, size=12),
    showlegend=False, hoverinfo='skip', cliponaxis=False,
))

# conceded_points is negative, so it stacks left of zero — pad the axis on both sides.
left = min(0, plot_df['conceded_points'].min()) * 1.3
right = plot_df['xPoints'].max() * 1.15

fig.update_layout(
    barmode='stack',
    bargap=0.35,
    title=dict(text='Top 10 defenders by expected points', font=dict(color=INK_PRIMARY, size=18)),
    legend=dict(font=dict(color=INK_SECONDARY)),
    plot_bgcolor=SURFACE,
    paper_bgcolor=SURFACE,
    font=dict(family='system-ui, -apple-system, "Segoe UI", sans-serif', color=INK_SECONDARY),
    xaxis=dict(title='Expected points', gridcolor=GRIDLINE, zerolinecolor=AXIS,
               tickfont=dict(color=INK_MUTED), range=[left, right]),
    yaxis=dict(tickfont=dict(color=INK_PRIMARY), automargin=True),
    margin=dict(l=10, r=150, t=60, b=40),
    height=520,
)
fig.show()

## Table view
Same data as the chart, for anyone who wants exact figures rather than reading bar lengths.

In [6]:
cols = ['web_name', 'team_name'] + [c for c, _, _ in COMPONENTS] + ['xPoints']
defenders.sort_values('xPoints', ascending=False)[cols].round(1)

,web_name,team_name,goal_points,assist_points,clean_points,defcon_points,appearance_points,bonus_points,conceded_points,xPoints
0,Gabriel,Arsenal,17.6,5.2,70.8,25.7,62.0,30.0,-2.4,209.0
241,Virgil,Liverpool,22.6,4.3,60.0,34.0,76.0,10.0,-8.0,198.9
373,Senesi,Spurs,9.2,14.2,38.5,45.8,74.0,14.0,-6.7,188.9
271,Guéhi,Man City,24.3,7.1,44.2,19.3,70.0,14.0,-4.6,174.4
371,Van Hecke,Spurs,19.9,4.7,37.5,34.8,72.0,8.0,-6.6,170.4
52,Truffert,Bournemouth,8.1,9.2,43.5,21.5,76.0,19.0,-8.9,168.3
180,Tarkowski,Everton,15.2,6.3,27.3,41.6,74.0,12.0,-8.8,167.6
1,J.Timber,Arsenal,28.3,4.6,63.1,2.7,56.0,9.0,-2.2,161.5
325,Thiaw,Newcastle,28.8,2.9,31.2,26.0,67.0,12.0,-7.8,160.1
154,Lacroix,Crystal Palace,14.7,2.5,25.3,43.7,69.0,11.0,-6.3,159.8
